# Attribute extraction pipeline
## Data-driven extraction

In [3]:
from google import genai
from google.genai import types
import os
client = genai.Client(api_key=os.environ["GENAI_API_KEY"])

In [5]:
import json 
def get_json(string):
    stripped = ""
    braces = 0
    for s in string:
        if s == "{":
            braces += 1
        if braces >= 1:
            stripped += s
        if s == "}":
            braces -= 1
    return json.loads(stripped, strict=False)


In [6]:
def retry(func, *args, retries=1, delay=30, **kwargs):
    """
    Call func with retries and delay on Exception.
    """
    for attempt in range(retries + 1):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if attempt == retries:
                # Last attempt failed, re-raise
                raise
            print(f"Error: {e}. {func} Another attempt...")
            time.sleep(delay)

In [4]:
dataset_name = "../data/categorization/DKK-dev-final.csv" 

In [7]:
import pandas as pd
import csv
import sys
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    dataset_name,
    sep=",",
    quoting=1,  # csv.QUOTE_ALL
    quotechar='"',
    escapechar="\\",
    engine="python"  # Required when dealing with multiline quoted fields
)

# Check the first few rows
marenky = df[df["judgement_model_short"] == "A"]
pod_vlivem = df[df["judgement_model_short"] == "B"]
s_nasledkem = df[df["judgement_model_short"] == "C"]
jina = df[df["judgement_model_short"] == "D"]
nedopravni = df[df["judgement_model_short"] == "E"]

In [8]:
print(f"Mařenka: {len(marenky)}")
print(f"Řízení pod vlivem omamných a psychotropních látek bez následku: {len(pod_vlivem)}")
print(f"Dopravní kriminalita s následkem : {len(s_nasledkem)}")
print(f"Dopravní kriminalita jiná: {len(jina)}")
print(f"Není dopravní kriminalita: {len(nedopravni)}")


Mařenka: 38
Řízení pod vlivem omamných a psychotropních látek bez následku: 156
Dopravní kriminalita s následkem : 92
Dopravní kriminalita jiná: 1
Není dopravní kriminalita: 895


## Řízení pod vlivem

In [26]:
def induce_new_attriutes(attribute_list):
    chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(system_instruction=f"""These are the residual texts from court verdicts after extracting the following attributes
    {attribute_list}

            What other pieces of information do you see repeatedly in the residual texts? Return a JSON with the emergent attribute names as keys and descriptions as values. The attributes should be as granular as possible, but do not repeat the attributes that were already defined.
     """),
       )
    response = chat.send_message(str(RESIDUALS))
    print(get_json(response.text))
    return get_json(response.text)

In [27]:
def populate_new_attributes(id, sentence, attribute_list):
    chat = client.chats.create(model="gemini-2.5-flash", config=types.GenerateContentConfig(system_instruction=f"""
        You extract values for attributes from text. Return only information from the text, never make anything up. If the value for attribute is not provided, return value 'n/a'. The date format is DD.MM.YYYY.  The time format is HH:MM. Decimal numbers are rounded to two digits after the decimal point: \#.\#\#. Format your output as JSON. The values mut be in the original language - Slovak. Extract the following attributes:
    {attribute_list}
"""),)
                              
    response = chat.send_message(sentence)
    new_attributes = get_json(response.text)
    
    POD_VLIVEM_STRUCTURED[id] = {**POD_VLIVEM_STRUCTURED[id], **new_attributes}   
      
    response = chat.send_message("What other information was there in the statement but is not covered by the attributes? Write one sentence containing the extra information. Output a JSON with one attribute: 'residual' where the value is the sentence with extra information. Only output information from the original text. The sentence mut be in the original language - Slovak.")
    POD_VLIVEM_STRUCTURED[id]["residual"] = get_json(response.text)["residual"]
    RESIDUALS[id] = get_json(response.text)["residual"]
    print({**new_attributes, "residual": RESIDUALS[id]})



In [28]:
import numpy as np
def check_population(candidate_attributes):
    final_attributes = {}
    df = pd.DataFrame.from_dict(POD_VLIVEM_STRUCTURED, orient="index").reset_index()
    df.replace("n/a", np.nan, inplace=True)
    nans = df.isna().sum()/len(df)
    for attribute in candidate_attributes:
        if nans[attribute] <= 0.85:
            final_attributes[attribute] = candidate_attributes[attribute]
            print(f"New attribute: {attribute}")
        else:
            print(f"Dropping attribute: {attribute}")
            df.drop(attribute, axis=1)
    
    return final_attributes
    

In [29]:
POD_VLIVEM_STRUCTURED = {}
RESIDUALS = {}
N = 20

In [30]:
# For full data-driven method, starting from a single pre-defined attribute: dateOfOffense
attribute_list = {"dateOfOffense": "Date when the offense occurred"}

# For a combination of a data-driven and a exper-based method, starting from a list of pre-defined attributes 
#attribute_list = {
#    "substanceUsed": "Drug or substance detected",
#    "amountUsed": "The amount of drug or substance detected",
#    "measuringDevice": "Measuring device used for testing",
#    "dateOfOffense": "Date when the offense occurred",
#    "timeOfOffense": "Time when the offense occurred",
#    "placeOfOffense": "Place where the offense occurred",
#    "checkedBy": "Who conducted the inspection or check",
#    "typeOfRoad": "Type of road where the offense occurred",
#    "reasonForCheck": "Reason why the inspection was carried out"
}


In [35]:
import time
for i in range(len(pod_vlivem)):     
    pod_vlivem_i = pod_vlivem.iloc[i]
    id = pod_vlivem_i["judgement_slt_id"]
    sentence = pod_vlivem_i["judgement_factual_sentence"]
    POD_VLIVEM_STRUCTURED[id] = {}
    POD_VLIVEM_STRUCTURED[id]["sentence"] = sentence
    print(i)
    print(pod_vlivem_i)
    retry(populate_new_attributes, id, sentence, attribute_list)
    if i % N == 0 and i!= 0:
        candidate_attributes = retry(induce_new_attriutes, attribute_list)
        for id, case in POD_VLIVEM_STRUCTURED.items():
            residual = case["residual"]
            retry(populate_new_attributes, id, residual, candidate_attributes)

        new_attributes = check_population(candidate_attributes)
        attribute_list = {**attribute_list, **new_attributes}

0
judgement_slt_id                                                        2276619
judgement_model_short                                                         B
judgement_factual_sentence    dňa 05.02.2018 v čase o 21.08 hod. po predchád...
Name: 38, dtype: object
{'dateOfOffense': '05.02.2018', 'residual': 'V čase o 21.08 hod. po predchádzajúcom požití alkoholických nápojov viedol vodič osobné motorové vozidlo zn. Citro?n Saxo, ev. č. M. XXXDD po ceste II/583 po ulici sv. Martina v obci Terchová v smere jazdy od obce Terchová Vyšné Kamence na časť obce Terchová - biely Potok, pričom bol ovplyvnený alkoholom vo výške 0,84 mg etanolu na liter vydychovaného vzduchu (čo predstavuje 1,75 promile) a svojim konaním porušil ustanovenia § 4 ods. 2 písm. d) Zákona č. 8/2009 Z.z. o cestnej premávke.'}
1
judgement_slt_id                                                        1808110
judgement_model_short                                                         B
judgement_factual_sentence    , že

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
df_pod_vlivem = pd.DataFrame.from_dict(POD_VLIVEM_STRUCTURED, orient="index").reset_index()
df_pod_vlivem.to_csv("DKK-dev_final_attributes.csv", index=False)

In [391]:
df_pod_vlivem

,index,substanceUsed,avgAmountUsed,avgAmountUsedUnit,amountUsed,measuringMethod,measuringTime,measuringDevice,refusedTesting,dateOfOffense,...,typeOfRoad,vehicleModel,vehicleType,checkedBy,reasonForCheck,reasonForCheckCategory,licenseStatus,lawBroken,note,sentence
0,95214574,alcohol,0.58,mg/l,0.64 mg/l; 0.55 mg/l; 0.54 mg/l,breath; breath; breath,03:52; 04:24; 04:54,"AlcoQuant 6020 plus, výrobné číslo A409530",False,01.01.2022,...,class_II,VW Golf,car,hliadkou ODI OR PZ Veľký Krtíš,zastavená a kontrolovaná,traffic_check,valid,NaN,Vozidlo malo evidenčné číslo VK 823CM a smer j...,- dňa 01.01.2022 v čase o 03.50 h viedla pod v...
1,95255832,alcohol,0.74,mg/l,0.74 mg/l,breath,NaN,NaN,False,01.01.2022,...,urban,Seat Altea,car,Obvodné oddelenie Policajného zboru Zlaté Moravce,Kontrola vozidla a vodiča,traffic_check,valid,NaN,Vozidlo malo evidenčné číslo LV-XXX EK.,že : po predchádzajúcom požití väčšieho množs...
2,95309650,methamphetamine; amphetamine,NaN,NaN,198.05 ng/ml; 33.56 ng/ml,blood; blood,10:30; 10:30,NaN,False,13.05.2021,...,urban,Dacia Logan,car,policajnou hliadkou,javil známky požitia omamných a psychotropných...,suspected_intoxication,valid,NaN,Podľa znaleckého posudku z odvetvia psychiatri...,", ž e dňa 13. mája 2021 v čase okolo 09.50 ho..."
3,95331065,alcohol,0.26,mg/l,0.26 mg/l,breath,NaN,NaN,False,04.01.2022,...,class_I,anonymized,car,hliadkou ODI Stará Ľubovňa,NaN,traffic_check,banned,NaN,Zákaz činnosti viesť motorové vozidlá na dobu ...,dňa 04.01.2022 v čase okolo 10.11 hod. pod vpl...
4,114902217,cannabis,44.50,ng/ml,7.00 ng/ml; 82.00 ng/ml,blood; blood,23:45; 23:45,NaN,False,19.11.2021,...,class_II,Škoda Octavia,car,OO PZ Čadca,"Kontrolovaný hliadkou OO PZ Čadca, nakoľko jav...",suspected_intoxication,valid,NaN,Vodič sa podrobil dychovej skúške na zistenie ...,dňa 19.11.2021 okolo 21:50 h viedol osobné mot...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1770,101536112,alcohol,0.59,mg/l,0.59 mg/l,breath,03:51,NaN,False,03.04.2022,...,urban,Mitsubishi Carisma,car,policajnou hliadkou,na zistenie požitia alkoholu,suspected_intoxication,valid,§ 4 odsek 2 písmeno d) zákona č. 8/2009 Z.z. o...,Evidenčné číslo vozidla bolo SE-XXXBE a jeho m...,že : dňa 03.04.2022 v čase o 03.51 hod. na ul...
1771,106473883,alcohol,0.61,mg/l,0.54 mg/l; 0.66 mg/l; 0.62 mg/l,breath; breath; breath,16:45; 17:06; 17:25,AlcoQuant 6020 plus,False,11.06.2022,...,NaN,Babeta,motorcycle,Okresný dopravný inšpektorát Trnava,jazda na motocykli bez ochrannej prilby a zo s...,traffic_violation; suspected_intoxication,valid,NaN,Sériové číslo prístroja AlcoQuant 6020 plus bo...,", ž e dňa 11.06.2022 v čase o 16:35 hodine vi..."
1772,108365243,alcohol,0.88,mg/l,0.89 mg/l; 0.82 mg/l; 0.95 mg/l; 0.86 mg/l,breath; breath; breath; breath,21:02; 21:17; 21:36; 22:06,AlcoQuant 6020 plus,False,18.06.2022,...,NaN,Mercedes Benz Vito,van,hliadkou OO PZ Partizánske,predpísaným spôsobom zastavený a kontrolovaný,traffic_check,valid,NaN,Vodič nemal pri sebe doklady potrebné na veden...,: dňa 18.06.2022 v čase o 20:50 h viedol osob...
1773,113761793,alcohol,1.22,mg/l,1.22 mg/l,breath,NaN,NaN,False,19.09.2022,...,urban,anonymized,car,OVS ODI OR PZ Košice,prekročenie povolenej rýchlosti,speeding,valid,NaN,Okrem extrahovaných údajov text uvádza hodnotu...,v Košiciach po ul. Gemerskej k ul. Rastislavov...
